In [ ]:
/**
 * ============================================================
 * CONSISTENCY MODELS
 * ============================================================
 *
 * Distributed system mein same data ki multiple copies hoti hain:
 *
 *     Primary DB
 *     Read Replicas
 *     Cache
 *     Search Index
 *     Vector Index
 *     Regional Replicas
 *
 *
 * Replication ke benefits:
 *
 *     - Higher availability
 *     - More read capacity
 *     - Lower geographic latency
 *     - Fault tolerance
 *
 *
 * But ek important problem create hoti hai:
 *
 *     "Write successful hone ke baad,
 *      later read ko kya value return karni chahiye?"
 *
 *
 * Is rule ko CONSISTENCY MODEL kehte hain.
 *
 *
 * IMPORTANT:
 *
 * Consistency ka matlab ye nahi hai ki data
 * business-wise correct hai.
 *
 * It only defines:
 *
 *     "Storage system reads/writes ko
 *      kis order/visibility ke saath expose karega?"
 *
 *
 * ============================================================
 * 1. WHY CONSISTENCY IS HARD
 * ============================================================
 *
 * Single machine:
 *
 *     Client
 *       |
 *       v
 *     Database
 *
 *     One copy
 *     One storage engine
 *     Easier ordering
 *
 *
 * Distributed system:
 *
 *     Client
 *       |
 *       v
 *     Node 1
 *       |
 *       +-------> Node 2
 *       |
 *       +-------> Node 3
 *
 *
 * Ab different problems ho sakti hain:
 *
 *     1. Write ek replica par pehle pahunch sakti hai.
 *
 *     2. Read stale replica se aa sakti hai.
 *
 *     3. Network partition replication rok sakta hai.
 *
 *     4. Multiple regions same item ko simultaneously
 *        update kar sakti hain.
 *
 *     5. Cache/search/vector index source DB se
 *        behind ho sakta hai.
 *
 *
 * ============================================================
 * 2. SIMPLE REPLICATION PROBLEM
 * ============================================================
 *
 * Suppose:
 *
 *     X = 3
 *
 *
 * Client:
 *
 *     Write X = 5
 *
 *
 * Flow:
 *
 *     Client
 *       |
 *       | Write X = 5
 *       v
 *     Node 1
 *       |
 *       +-----------------> Node 3
 *       |                    X = 5
 *       |
 *       +---- delayed -----> Node 2
 *                            X = 3
 *                            (stale)
 *
 *
 * Ab another client Node 2 se read karta hai.
 *
 * Question:
 *
 *     Should it return:
 *
 *         X = 3
 *
 *     OR
 *
 *         Wait until Node 2 catches up
 *
 *     OR
 *
 *         Read another replica
 *
 *     OR
 *
 *         Reject the read?
 *
 *
 * Different consistency models ka answer
 * different hota hai.
 *
 *
 * ============================================================
 * 3. CONSISTENCY IS A CONTRACT
 * ============================================================
 *
 * Consistency model basically define karta hai:
 *
 *     Given a history of writes,
 *     which values can a read return?
 *
 *
 * Therefore:
 *
 *     "Database consistent hai"
 *
 * is not a useful complete statement.
 *
 * Instead ask:
 *
 *     "Exactly kya guarantee mil rahi hai?"
 *
 *
 * Examples:
 *
 *     Linearizability
 *     Sequential Consistency
 *     Strict Serializability
 *     Causal Consistency
 *     Read-Your-Writes
 *     Monotonic Reads
 *     Eventual Consistency
 *
 *
 * ============================================================
 * 4. CONSISTENCY SPECTRUM
 * ============================================================
 *
 * Generally:
 *
 *     WEAKER
 *       |
 *       v
 *     Eventual
 *     FIFO / PRAM
 *     Causal
 *     Client-centric guarantees
 *     Sequential
 *     Linearizable
 *     Strict Serializability
 *       |
 *       v
 *     STRONGER
 *
 *
 * General rule:
 *
 *     Stronger guarantee
 *          |
 *          v
 *     More coordination
 *          |
 *          v
 *     More latency / complexity
 *
 *
 * Weaker guarantee:
 *
 *     Less coordination
 *          |
 *          v
 *     Better availability / latency
 *          |
 *          v
 *     More possibility of stale data
 *
 *
 * IMPORTANT:
 *
 * Some models are not strictly comparable.
 * For example:
 *
 *     Read-Your-Writes
 *     Monotonic Reads
 *
 * are mainly client/session guarantees,
 * while Linearizability is a system-wide guarantee.
 *
 *
 * ============================================================
 * 5. COORDINATION COST
 * ============================================================
 *
 * Strong consistency generally requires
 * more coordination between nodes.
 *
 *
 * Example:
 *
 *     Client
 *       |
 *       v
 *     Node 1
 *       |
 *       +-------> Node 2
 *       |
 *       +-------> Node 3
 *       |
 *       v
 *     Confirm
 *       |
 *       v
 *     Client
 *
 *
 * Benefit:
 *
 *     Latest state ka confidence higher.
 *
 *
 * Cost:
 *
 *     - Higher latency
 *     - More network communication
 *     - Lower availability during partitions
 *     - Leader bottleneck
 *     - More conflicts
 *     - More complex failover
 *
 *
 * Weaker consistency:
 *
 *     Client
 *       |
 *       v
 *     Local Replica
 *       |
 *       v
 *     Fast response
 *
 *
 * But:
 *
 *     Replica stale ho sakti hai.
 *
 *
 * ============================================================
 * 6. RIGHT CONSISTENCY MODEL
 * ============================================================
 *
 * Golden rule:
 *
 *     Choose the WEAKEST consistency model
 *     that still protects correctness.
 *
 *
 * Don't use strongest consistency everywhere.
 *
 *
 * Example:
 *
 *     Payment
 *         -> Strong
 *
 *     Inventory
 *         -> Strong
 *
 *     Search
 *         -> Eventual
 *
 *     Recommendations
 *         -> Eventual
 *
 *     Analytics
 *         -> Eventual
 *
 *
 * ============================================================
 * 7. LINEARIZABILITY
 * ============================================================
 *
 * Linearizability is a strong consistency model.
 *
 * Simple meaning:
 *
 *     System behave kare jaise
 *     ek hi latest authoritative copy ho.
 *
 *
 * Main rule:
 *
 *     If WRITE completes before READ starts,
 *     READ must see that WRITE or a newer value.
 *
 *
 * Example:
 *
 *     Initial:
 *
 *         X = 0
 *
 *     Client 1:
 *
 *         Write X = 1
 *              |
 *              v
 *             OK
 *
 *     Client 2:
 *
 *         Read X
 *
 *     Result:
 *
 *         X = 1
 *
 *
 * NOT:
 *
 *         X = 0
 *
 *
 * ============================================================
 * 8. LINEARIZATION POINT
 * ============================================================
 *
 * Har operation ko imagine karo ki
 * ek exact point par atomically execute hui.
 *
 *
 *     Request
 *        |
 *        |
 *        v
 *     +---------+
 *     | Write   |
 *     +---------+
 *        |
 *        X
 *   Linearization
 *      Point
 *        |
 *        v
 *     Response
 *
 *
 * Operation ke request-response interval ke
 * andar ek point hota hai jahan operation
 * logically take effect karti hai.
 *
 *
 * ============================================================
 * 9. LINEARIZABILITY GUARANTEES
 * ============================================================
 *
 * 1. Real-time ordering
 *
 *     A complete hua
 *          |
 *          v
 *     B start hua
 *
 *     B ko A ya uske baad ka state dekhna hoga.
 *
 *
 * 2. Atomic visibility
 *
 *     Operation ek point par visible hoti hai.
 *
 *
 * 3. Single-object clarity
 *
 *     Client replicated object ko
 *     ek single copy ki tarah reason kar sakta hai.
 *
 *
 * ============================================================
 * 10. LINEARIZABILITY IMPLEMENTATION
 * ============================================================
 *
 * Common techniques:
 *
 *     - Leader
 *     - Consensus
 *     - Quorum
 *     - Leader-freshness verification
 *
 *
 * Example:
 *
 *     Client
 *       |
 *       v
 *     Leader
 *       |
 *       +-------> Replica 1
 *       |
 *       +-------> Replica 2
 *
 *
 * Follower ke disk par stale data ho sakta hai.
 *
 * Strong read guarantee maintain karne ke liye
 * system unsafe follower read ko serve nahi karega.
 *
 *
 * ============================================================
 * 11. LINEARIZABILITY USE CASES
 * ============================================================
 *
 * Use when stale data correctness bug create kar sakta hai.
 *
 * Examples:
 *
 *     - Distributed locks
 *     - Leader election metadata
 *     - Account balance reads
 *     - Inventory reservation
 *     - Quota counters
 *     - Permission changes
 *     - Idempotency records
 *     - Critical state transitions
 *
 *
 * AI systems mein:
 *
 *     - Billing usage
 *     - Rate limits
 *     - Model rollout state
 *     - Access control
 *     - Prompt safety policy versions
 *     - Long-running job ownership
 *
 *
 * Usually unnecessary for:
 *
 *     - Recommendations
 *     - Embedding lookup
 *     - Analytics counters
 *
 *
 * ============================================================
 * 12. SEQUENTIAL CONSISTENCY
 * ============================================================
 *
 * Sequential consistency linearizability se weaker hai.
 *
 *
 * Rule:
 *
 *     All operations kisi ek sequential order
 *     mein appear honi chahiye.
 *
 *
 * BUT:
 *
 *     Global order ko real-time order follow
 *     karna mandatory nahi hai.
 *
 *
 * Important:
 *
 *     Each client's own operation order
 *     preserve hona chahiye.
 *
 *
 * Example:
 *
 *     Client 1:
 *
 *         Write X = 1
 *         Read Y = 0
 *
 *
 *     Client 2:
 *
 *         Write Y = 1
 *         Read X = 1
 *
 *
 * Ye ek valid sequential ordering se explain
 * ho sakta hai, as long as each client's
 * local order preserved hai.
 *
 *
 * ============================================================
 * 13. LINEARIZABLE vs SEQUENTIAL
 * ============================================================
 *
 *
 * LINEARIZABLE:
 *
 *     Real-time order MUST be respected.
 *
 *
 *     Write completes
 *          |
 *          v
 *     Read starts
 *          |
 *          v
 *     Read must see write
 *
 *
 * SEQUENTIAL:
 *
 *     Single global sequential order hona chahiye
 *     but real-time order necessarily preserve
 *     nahi karna.
 *
 *
 * Memory trick:
 *
 *     Linearizable = Sequential + Real-Time
 *
 *
 * ============================================================
 * 14. SERIALIZABILITY
 * ============================================================
 *
 * Serializability transactions ke liye
 * isolation guarantee hai.
 *
 *
 * Rule:
 *
 *     Concurrent transactions ka result
 *     kisi serial execution ke equivalent hona chahiye.
 *
 *
 * Example:
 *
 *     Transaction A
 *     Transaction B
 *
 * simultaneously execute kar sakti hain,
 * but final result should look like:
 *
 *     A -> B
 *
 * OR
 *
 *     B -> A
 *
 *
 * as if they executed one after another.
 *
 *
 * ============================================================
 * 15. SERIALIZABILITY != LINEARIZABILITY
 * ============================================================
 *
 * Serializability:
 *
 *     Transaction correctness.
 *
 * Real-time ordering necessarily required nahi.
 *
 *
 * Strict Serializability:
 *
 *     Serializability
 *          +
 *     Real-time ordering
 *
 *
 * Example:
 *
 *     Transaction A commits
 *          |
 *          v
 *     Transaction B starts
 *
 *
 * Strict serializability:
 *
 *     A must appear before B.
 *
 *
 * ============================================================
 * 16. IMPORTANT COMPARISON
 * ============================================================
 *
 *     Linearizability
 *         |
 *         +--> Single-object operations
 *         |
 *         +--> Real-time order YES
 *
 *
 *     Serializability
 *         |
 *         +--> Transactions
 *         |
 *         +--> Real-time order NOT necessarily
 *
 *
 *     Strict Serializability
 *         |
 *         +--> Transactions
 *         |
 *         +--> Real-time order YES
 *
 *
 * ============================================================
 * 17. STRICT SERIALIZABILITY USE CASES
 * ============================================================
 *
 * Strong transaction correctness required:
 *
 *     - Financial transfers
 *     - Seat booking
 *     - Scarce inventory
 *     - Order placement
 *     - Entitlement grants
 *     - Billing updates
 *     - Multi-row invariants
 *     - Multi-object correctness
 *
 *
 * ============================================================
 * 18. EVENTUAL CONSISTENCY
 * ============================================================
 *
 * Eventual consistency is a weak consistency model.
 *
 *
 * Main promise:
 *
 *     If updates stop
 *     AND replication succeeds,
 *
 *     eventually all replicas converge
 *     to the same value.
 *
 *
 * Example:
 *
 *     Write X = 5
 *
 *     Replica 1 -> X = 5
 *     Replica 2 -> X = 0
 *     Replica 3 -> X = 0
 *
 *
 *     Later:
 *
 *     Replica 1 -> X = 5
 *     Replica 2 -> X = 5
 *     Replica 3 -> X = 5
 *
 *
 * ============================================================
 * 19. WHAT EVENTUAL CONSISTENCY DOES NOT PROMISE
 * ============================================================
 *
 * It does NOT guarantee:
 *
 *     - When convergence will happen
 *     - Latest write immediately visible
 *     - Bounded staleness
 *     - Read-your-writes
 *     - Monotonic reads
 *     - Conflict resolution
 *     - Durability
 *
 *
 * VERY IMPORTANT:
 *
 *     Eventual consistency
 *     !=
 *     Durable
 *
 *
 *     Eventual consistency
 *     !=
 *     Automatically available
 *
 *
 * It mainly describes:
 *
 *     Visibility + Convergence
 *
 *
 * ============================================================
 * 20. EVENTUAL CONSISTENCY EXAMPLE
 * ============================================================
 *
 *     Writer
 *       |
 *       v
 *     Replica 1
 *       |
 *       +---- delayed ----> Replica 2
 *       |
 *       +---- delayed ----> Replica 3
 *
 *
 * Immediately:
 *
 *     R1 = 5
 *     R2 = 0
 *     R3 = 0
 *
 *
 * Later:
 *
 *     R1 = 5
 *     R2 = 5
 *     R3 = 5
 *
 *
 * Convergence achieved.
 *
 *
 * ============================================================
 * 21. CONFLICT RESOLUTION
 * ============================================================
 *
 * Eventual consistency mein multiple replicas
 * independently write kar sakti hain.
 *
 *
 * Example:
 *
 *     Replica 1:
 *
 *         Name = "Mohammad"
 *
 *     Replica 2:
 *
 *         Name = "Mohd"
 *
 *
 * Network partition ke baad:
 *
 *     Conflict!
 *
 *
 * System ko decide karna padega:
 *
 *     Which value wins?
 *
 *
 * ============================================================
 * 22. LAST-WRITE-WINS (LWW)
 * ============================================================
 *
 * Rule:
 *
 *     Highest timestamp/version wins.
 *
 *
 * Example:
 *
 *     Write A:
 *         X = 10
 *         timestamp = 100
 *
 *     Write B:
 *         X = 20
 *         timestamp = 101
 *
 *
 * Result:
 *
 *     X = 20
 *
 *
 * Problem:
 *
 *     Valid write silently lose ho sakti hai.
 *
 *
 * Good fit:
 *
 *     - Ephemeral presence
 *     - Cache refresh
 *
 *
 * Risky for:
 *
 *     - Profiles
 *     - Documents
 *     - Shopping carts
 *     - User data
 *
 *
 * ============================================================
 * 23. VERSION VECTORS / VECTOR CLOCKS
 * ============================================================
 *
 * Purpose:
 *
 *     Causality track karna
 *     aur concurrent updates detect karna.
 *
 *
 * BUT:
 *
 *     Conflict detect karne ke baad bhi
 *     application ko merge logic ki
 *     zarurat ho sakti hai.
 *
 *
 * ============================================================
 * 24. CRDT
 * ============================================================
 *
 * CRDT =
 *
 *     Conflict-free Replicated Data Type
 *
 *
 * Idea:
 *
 *     Aise data types design karo jo
 *     coordination ke bina safely merge ho sakein.
 *
 *
 * Benefit:
 *
 *     Automatic merge possible.
 *
 *
 * Limitation:
 *
 *     Har type of operation/data ke liye
 *     suitable nahi hota.
 *
 *
 * ============================================================
 * 25. APPLICATION-LEVEL MERGE
 * ============================================================
 *
 * Business logic decide kare:
 *
 *     Region A + Region B
 *             |
 *             v
 *       Custom merge
 *
 *
 * Benefit:
 *
 *     Business correctness preserve kar sakte hain.
 *
 *
 * Cost:
 *
 *     Build aur test karna difficult.
 *
 *
 * ============================================================
 * 26. REJECT ON CONFLICT
 * ============================================================
 *
 * Conflict detect karo:
 *
 *     Version mismatch
 *          |
 *          v
 *     Reject
 *          |
 *          v
 *     Client retries
 *
 *
 * Benefit:
 *
 *     Silent data loss avoid hota hai.
 *
 *
 * Cost:
 *
 *     User ko failures/retries dikh sakte hain.
 *
 *
 * ============================================================
 * 27. WHERE EVENTUAL CONSISTENCY WORKS WELL
 * ============================================================
 *
 * Good use cases:
 *
 *     CDN caches
 *     DNS
 *     Like/view counters
 *     Search indexes
 *     Vector indexes
 *     Feature stores
 *     Analytics
 *     Activity feeds
 *     Notifications
 *     Recommendations
 *
 *
 * Why?
 *
 *     Slight delay usually acceptable hai.
 *
 *
 * Example:
 *
 *     Document uploaded
 *          |
 *          v
 *     Embedding created
 *          |
 *          v
 *     Vector index updated
 *
 * User ko immediately search result na mile,
 * generally acceptable ho sakta hai.
 *
 *
 * ============================================================
 * 28. WHERE EVENTUAL CONSISTENCY IS DANGEROUS
 * ============================================================
 *
 * Avoid for hard invariants:
 *
 *     Inventory reservation
 *     Payment authorization
 *     Access control
 *     Safety policy updates
 *     Exclusive job ownership
 *
 *
 * Example:
 *
 *     Stock = 1
 *
 *     Region A -> Reserve SUCCESS
 *     Region B -> Reserve SUCCESS
 *
 *     Result:
 *
 *         1 item
 *         2 customers
 *
 *     BAD!
 *
 *
 * ============================================================
 * 29. SOURCE OF TRUTH vs DERIVED DATA
 * ============================================================
 *
 * Important system-design rule:
 *
 *
 *     SOURCE OF TRUTH
 *         |
 *         +--> Strong consistency
 *
 *
 *     DERIVED DATA
 *         |
 *         +--> Weaker consistency
 *
 *
 * Example:
 *
 *     Primary DB
 *         |
 *         +-----> Search Index
 *         |
 *         +-----> Vector Index
 *         |
 *         +-----> Analytics
 *         |
 *         +-----> Recommendations
 *
 *
 * Primary DB:
 *
 *     Stronger guarantees
 *
 *
 * Derived systems:
 *
 *     Eventual consistency often acceptable.
 *
 *
 * ============================================================
 * 30. CAUSAL CONSISTENCY
 * ============================================================
 *
 * Causal consistency preserves:
 *
 *     CAUSE -> EFFECT
 *
 *
 * If:
 *
 *     B depends on A
 *
 * Then anyone who sees B must have seen A first.
 *
 *
 * Example:
 *
 *     Alice:
 *
 *         "Let's meet at 5 PM"
 *
 *              |
 *              v
 *     Bob reads Alice's message
 *
 *              |
 *              v
 *     Bob:
 *
 *         "Sounds good"
 *
 *
 * Bob's reply depends on Alice's message.
 *
 *
 * Therefore:
 *
 *     Alice's message
 *          |
 *          v
 *     Bob's reply
 *
 *
 * Must NOT appear as:
 *
 *     Bob's reply
 *          |
 *          v
 *     Alice's message
 *
 *
 * ============================================================
 * 31. SOURCES OF CAUSALITY
 * ============================================================
 *
 * 1. Program order
 *
 *     A happens before B
 *     in the same client's execution.
 *
 *
 * 2. Read-from relationship
 *
 *     Client reads A
 *        |
 *        v
 *     Client writes B based on A
 *
 *
 * 3. Transitivity
 *
 *     A -> B
 *     B -> C
 *
 *     Therefore:
 *
 *     A -> C
 *
 *
 * ============================================================
 * 32. CAUSAL CONSISTENCY USE CASES
 * ============================================================
 *
 *     - Social feeds
 *     - Comment threads
 *     - Chat
 *     - Collaborative documents
 *     - Issue trackers
 *     - Code review
 *     - Notebook collaboration
 *     - AI assistant conversations
 *
 *
 * Especially useful when:
 *
 *     User ko cause-effect order naturally
 *     expected hota hai.
 *
 *
 * ============================================================
 * 33. PRAM / FIFO CONSISTENCY
 * ============================================================
 *
 * PRAM = FIFO consistency.
 *
 *
 * Main guarantee:
 *
 *     Ek process ke writes ko
 *     every other process same order mein observe kare.
 *
 *
 * Example:
 *
 *     Process 1:
 *
 *         Write A
 *         Write B
 *
 *
 * Other processes:
 *
 *         A before B
 *
 * Must observe.
 *
 *
 * BUT:
 *
 * Different processes ke writes ko globally
 * order karna necessary nahi hai.
 *
 *
 * Also:
 *
 *     Read-based dependency preserve nahi hoti.
 *
 *
 * ============================================================
 * 34. PRAM vs CAUSAL
 * ============================================================
 *
 * Suppose:
 *
 *     Process 1:
 *         Write X = 1
 *
 *     Process 2:
 *         Read X = 1
 *         Write Y = 2
 *
 *
 * Causal consistency:
 *
 *     Anyone seeing Y = 2
 *     must also see X = 1.
 *
 *
 * PRAM:
 *
 *     Ye dependency guarantee nahi karta.
 *
 *
 * Therefore:
 *
 *     PRAM < Causal
 *
 * in terms of this dependency guarantee.
 *
 *
 * ============================================================
 * 35. CLIENT-CENTRIC CONSISTENCY
 * ============================================================
 *
 * Kuch guarantees poore distributed system ke
 * liye nahi, sirf ek client/session ke liye hoti hain.
 *
 *
 * Ye especially useful hain because:
 *
 *     User-facing bugs often session-related hote hain.
 *
 *
 * Example:
 *
 *     User changes profile photo
 *          |
 *          v
 *     Refresh
 *          |
 *          v
 *     Old photo
 *
 *
 * Other users ko maybe koi issue nahi,
 * but same user ko ye confusing lagega.
 *
 *
 * ============================================================
 * 36. READ-YOUR-WRITES
 * ============================================================
 *
 * Guarantee:
 *
 *     Client ne jo write kiya,
 *     usi client ke later reads ko
 *     woh write ya newer value dikhni chahiye.
 *
 *
 * Example:
 *
 *     Client
 *       |
 *       | Write X = 5
 *       v
 *     Replica 1
 *       |
 *       | replication in progress
 *       v
 *     Replica 2
 *
 *     Client later:
 *
 *       Read X
 *
 *     Must return:
 *
 *       X = 5
 *       OR newer
 *
 *     NOT:
 *
 *       X = 3
 *
 *
 * ============================================================
 * 37. READ-YOUR-WRITES IMPLEMENTATION
 * ============================================================
 *
 * Common techniques:
 *
 *     - Route reads to primary temporarily
 *     - Session tokens
 *     - Sticky session / replica pinning
 *     - Minimum version on reads
 *
 *
 * ============================================================
 * 38. READ-YOUR-WRITES USE CASES
 * ============================================================
 *
 *     - Profile update
 *     - Account settings
 *     - Uploaded files
 *     - Conversation messages
 *     - Prompt edits
 *     - Saved agents
 *     - Workspace changes
 *
 *
 * Simple rule:
 *
 *     "User ne save kiya hai,
 *      to usko apna change immediately dikhna chahiye."
 *
 *
 * ============================================================
 * 39. MONOTONIC READS
 * ============================================================
 *
 * Guarantee:
 *
 *     Once client ne koi value observe kar li,
 *     later read older value return nahi karega.
 *
 *
 * Example:
 *
 *     First read:
 *
 *         X = 5
 *
 *     Later another replica se read:
 *
 *         X = 3
 *
 *     NOT ALLOWED.
 *
 *
 * Later read must return:
 *
 *         X = 5
 *
 *     OR newer
 *
 *     OR wait.
 *
 *
 * ============================================================
 * 40. WITHOUT MONOTONIC READS
 * ============================================================
 *
 * User sees:
 *
 *     Read 1 -> X = 5
 *
 *     Refresh
 *
 *     Read 2 -> X = 3
 *
 *
 * Data appears to move BACKWARD in time.
 *
 *
 * This can cause:
 *
 *     - User retries operation
 *     - Support tickets
 *     - Loss of trust
 *     - "Data disappeared" feeling
 *
 *
 * Good use cases:
 *
 *     - Order history
 *     - Message threads
 *     - Audit logs
 *     - Document versions
 *     - Deployment history
 *     - Evaluation runs
 *
 *
 * ============================================================
 * 41. MONOTONIC WRITES
 * ============================================================
 *
 * Guarantee:
 *
 *     One client's writes are applied
 *     in that client's order.
 *
 *
 * Example:
 *
 *     Client:
 *
 *         Write A
 *         Write B
 *
 *
 * Replica must NOT apply:
 *
 *         B
 *         A
 *
 *
 * It must preserve:
 *
 *         A -> B
 *
 *
 * ============================================================
 * 42. WHY MONOTONIC WRITES MATTER
 * ============================================================
 *
 * Later writes may depend on earlier writes.
 *
 *
 * Example:
 *
 *     Model artifact upload
 *          |
 *          v
 *     Model configuration publish
 *
 *
 * Configuration ko artifact se pehle publish
 * kar diya:
 *
 *     BAD
 *
 *
 * Another example:
 *
 *     Migration complete flag
 *          |
 *          v
 *     Read new schema
 *
 *
 * Flag pehle visible ho gaya,
 * migration actually complete nahi hui:
 *
 *     BAD
 *
 *
 * ============================================================
 * 43. WRITES-FOLLOW-READS
 * ============================================================
 *
 * Guarantee:
 *
 *     Client ne value READ ki
 *          |
 *          v
 *     Client us value ke basis par WRITE karta hai
 *
 *
 * Then:
 *
 *     WRITE must be ordered after that READ.
 *
 *
 * Example:
 *
 *     Read version = 7
 *          |
 *          v
 *     Client edits document
 *          |
 *          v
 *     Write version = 8
 *
 *
 * System ko ensure karna hai ki write
 * actually version 7 ke state ke baad hai.
 *
 *
 * ============================================================
 * 44. WITHOUT WRITES-FOLLOW-READS
 * ============================================================
 *
 * Client:
 *
 *     Read X = 7
 *          |
 *          v
 *     Modify X
 *          |
 *          v
 *     Write
 *
 *
 * But system stale/incorrect ordering ki wajah se
 * aisa write accept kar sakta hai jo client ke
 * observed state par actually based nahi hai.
 *
 *
 * Useful for:
 *
 *     - Compare-and-swap
 *     - Conditional updates
 *     - Collaborative editing
 *     - Optimistic concurrency
 *     - Workflow transitions
 *     - AI annotation/review systems
 *
 *
 * ============================================================
 * 45. CLIENT-CENTRIC MODELS QUICK TABLE
 * ============================================================
 *
 * Read-Your-Writes:
 *
 *     "Mera write mujhe later read mein dikhe."
 *
 *
 * Monotonic Reads:
 *
 *     "Jo newer value dekh li,
 *      uske baad older value mat dikhao."
 *
 *
 * Monotonic Writes:
 *
 *     "Mere writes same order mein apply karo."
 *
 *
 * Writes-Follow-Reads:
 *
 *     "Mere read ke basis par hua write
 *      us read ke baad ordered ho."
 *
 *
 * ============================================================
 * 46. REAL DATABASES CONSISTENCY KO
 *     KAISE EXPOSE KARTE HAIN?
 * ============================================================
 *
 * Production database generally simply:
 *
 *     "Strong"
 *     "Weak"
 *
 * jaise buttons nahi deta.
 *
 *
 * Instead knobs milte hain:
 *
 *     - Read concern
 *     - Write concern
 *     - Quorum
 *     - Transaction isolation
 *     - Leader reads
 *     - Follower reads
 *     - Session guarantees
 *     - Global table modes
 *     - Cache TTL
 *
 *
 * IMPORTANT:
 *
 * "Is database consistent?"
 *
 * se better question:
 *
 *     "Is particular operation ke liye,
 *      acknowledged write ko kaunse reads,
 *      kaunse region se,
 *      kaunse failure conditions mein
 *      observe karenge?"
 *
 *
 * ============================================================
 * 47. COMMON IMPLEMENTATION PATTERNS
 * ============================================================
 *
 *
 * SINGLE LEADER + PRIMARY READS
 *
 *     Client
 *       |
 *       v
 *     Leader
 *
 *     Strong reads from leader.
 *
 *     Followers:
 *
 *         May lag.
 *
 *
 * ------------------------------------------------------------
 *
 * LEADER + ASYNC READ REPLICAS
 *
 *     Client
 *       |
 *       v
 *     Read Replica
 *
 *     Fast/scalable,
 *     but may return stale data.
 *
 *
 * ------------------------------------------------------------
 *
 * CONSENSUS GROUP
 *
 *     Node 1
 *       |
 *       +---- Node 2
 *       |
 *       +---- Node 3
 *
 *     Consensus:
 *
 *         Raft / Paxos
 *
 *     Can provide linearizable operations
 *     within the group.
 *
 *
 * ------------------------------------------------------------
 *
 * QUORUM READ + WRITE
 *
 *     N = total replicas
 *
 *     R = read quorum
 *     W = write quorum
 *
 *
 *     Common intuition:
 *
 *         R + W > N
 *
 *
 *     means read/write quorums overlap,
 *     which can provide stronger visibility
 *     under appropriate failure assumptions.
 *
 *
 * IMPORTANT:
 *
 *     R + W > N alone does not magically
 *     guarantee every possible consistency property.
 *     Failure handling and implementation matter.
 *
 *
 * ------------------------------------------------------------
 *
 * SESSION TOKENS
 *
 *     Useful for:
 *
 *         Read-your-writes
 *         Monotonic reads
 *
 *
 * ------------------------------------------------------------
 *
 * MULTI-REGION ACTIVE-ACTIVE
 *
 *     Region A <----> Region B
 *
 *     Benefit:
 *
 *         Low local latency
 *
 *     Cost:
 *
 *         Conflict resolution
 *         OR
 *         global coordination
 *
 *
 * ------------------------------------------------------------
 *
 * CACHE + TTL / INVALIDATION
 *
 *     Primary DB
 *          |
 *          v
 *        Cache
 *
 *     Visibility may be eventual.
 *
 *     Freshness depends on:
 *
 *         TTL
 *         Invalidation correctness
 *
 *
 * ------------------------------------------------------------
 *
 * EVENT-DRIVEN PROJECTION
 *
 *     Source DB
 *         |
 *         | Event
 *         v
 *     Consumer
 *         |
 *         v
 *     Derived View
 *
 *     Eventual consistency.
 *
 *     Freshness depends on consumer lag
 *     and replay health.
 *
 *
 * ============================================================
 * 48. DON'T ASSUME CONSISTENCY FROM DATABASE TYPE
 * ============================================================
 *
 * Don't say:
 *
 *     "NoSQL = Eventual"
 *
 * or
 *
 *     "SQL = Strong"
 *
 *
 * Actual guarantee depends on:
 *
 *     - Product
 *     - Deployment mode
 *     - Operation
 *     - Index type
 *     - Read/write configuration
 *     - Transaction mode
 *
 *
 * Example:
 *
 *     Key-value store:
 *         May support eventual reads
 *         + selected strong reads.
 *
 *
 *     Wide-column store:
 *         May support quorum levels.
 *
 *
 *     Document DB:
 *         May expose read/write concerns
 *         and sessions.
 *
 *
 *     Global SQL:
 *         Can provide strict transactions,
 *         but cross-region writes may have
 *         coordination latency.
 *
 *
 *     Search / Vector Index:
 *         Often derived from primary DB
 *         and intentionally lagging.
 *
 *
 * ============================================================
 * 49. HOW TO CHOOSE CONSISTENCY MODEL
 * ============================================================
 *
 * DON'T START WITH:
 *
 *     "Which database should I use?"
 *
 *
 * START WITH:
 *
 *     "What breaks if this read is stale?"
 *
 *
 * Ask:
 *
 *     1. What is the source of truth?
 *
 *     2. Which reads must observe acknowledged writes?
 *
 *     3. Is guarantee global, regional,
 *        per-session, per-object, etc.?
 *
 *     4. What happens during network partition?
 *
 *     5. Can two writers update same item?
 *
 *     6. If conflict occurs, who resolves it?
 *
 *     7. How much staleness is acceptable?
 *
 *     8. Does UI need pending/syncing status?
 *
 *     9. Are search/vector/cache/analytics
 *        clearly derived data?
 *
 *     10. Do tests cover stale reads,
 *         reordered events, retries,
 *         duplicate delivery and failover?
 *
 *
 * ============================================================
 * 50. CONSISTENCY DECISION TREE
 * ============================================================
 *
 *                 Can stale data
 *                  be tolerated?
 *                    /       \
 *                  NO         YES
 *                  |           |
 *                  v           v
 *            Strong model   Must user see
 *                           own changes?
 *                              /    \
 *                            YES     NO
 *                             |       |
 *                             v       v
 *                       Read-Your-  Must cause
 *                       Writes      & effect
 *                                   stay ordered?
 *                                      /   \
 *                                    YES    NO
 *                                     |      |
 *                                     v      v
 *                                   Causal Eventual
 *
 *
 * ============================================================
 * 51. USE CASE -> CONSISTENCY MODEL
 * ============================================================
 *
 * PAYMENT AUTHORIZATION
 *
 *     Strict Serializability
 *
 * Why?
 *
 *     Money invariants must be preserved.
 *
 *
 * ------------------------------------------------------------
 *
 * BANK BALANCE AFTER TRANSFER
 *
 *     Linearizable /
 *     Transactionally consistent
 *
 * Why?
 *
 *     Stale balance dangerous hai.
 *
 *
 * ------------------------------------------------------------
 *
 * SCARCE INVENTORY
 *
 *     Linearizable claim
 *     OR Serializable transaction
 *
 * Why?
 *
 *     Overselling prevent karna hai.
 *
 *
 * ------------------------------------------------------------
 *
 * DISTRIBUTED LOCK
 *
 *     Linearizability
 *
 * Why?
 *
 *     Two clients simultaneously lock
 *     hold nahi karne chahiye.
 *
 *
 * ------------------------------------------------------------
 *
 * PERMISSION REVOCATION
 *
 *     Strong read on enforcement path
 *
 * Why?
 *
 *     Stale permission security issue
 *     create kar sakti hai.
 *
 *
 * ------------------------------------------------------------
 *
 * USER PROFILE UPDATE
 *
 *     Read-Your-Writes
 *
 * Why?
 *
 *     User ko apna saved change dikhna chahiye.
 *
 *
 * ------------------------------------------------------------
 *
 * CHAT / COMMENTS
 *
 *     Causal Consistency
 *
 * Why?
 *
 *     Reply original message se pehle
 *     appear nahi hona chahiye.
 *
 *
 * ------------------------------------------------------------
 *
 * SEARCH
 *
 *     Eventual Consistency
 *     + measured lag
 *
 * Why?
 *
 *     Search index derived data hai.
 *
 *
 * ------------------------------------------------------------
 *
 * VECTOR RETRIEVAL
 *
 *     Eventual Consistency
 *     + ingestion status
 *
 * Why?
 *
 *     Embedding/indexing asynchronous hai.
 *
 *
 * ------------------------------------------------------------
 *
 * LIKE / VIEW COUNTS
 *
 *     Eventual / Approximate
 *
 * Why?
 *
 *     Exact freshness usually worth the cost nahi.
 *
 *
 * ------------------------------------------------------------
 *
 * ANALYTICS DASHBOARD
 *
 *     Eventual
 *     + freshness timestamp
 *
 * Why?
 *
 *     User ko pata hona chahiye data kitna old hai.
 *
 *
 * ------------------------------------------------------------
 *
 * MODEL EVALUATION METRICS
 *
 *     Eventual
 *     + run state
 *
 * Why?
 *
 *     Aggregates completed events se lag kar sakte hain.
 *
 *
 * ============================================================
 * 52. MIXED CONSISTENCY SYSTEM
 * ============================================================
 *
 * Real production system mein ek hi consistency
 * model everywhere use nahi hota.
 *
 *
 * Example AI Platform:
 *
 *
 *     +--------------------------------------------------+
 *     |                  AI PLATFORM                    |
 *     +--------------------------------------------------+
 *
 *     STRONG CONSISTENCY
 *     |
 *     +--> Billing
 *     +--> Quotas
 *     +--> Access Control
 *     +--> Model Deployment State
 *     +--> Exclusive Job Claims
 *
 *
 *     CAUSAL / SESSION CONSISTENCY
 *     |
 *     +--> Conversations
 *     +--> Prompt Editing
 *     +--> Human Review Workflow
 *
 *
 *     EVENTUAL CONSISTENCY
 *     |
 *     +--> Search Index
 *     +--> Vector Index
 *     +--> Usage Aggregates
 *     +--> Recommendations
 *
 *
 * ============================================================
 * 53. WHY NOT STRONG CONSISTENCY EVERYWHERE?
 * ============================================================
 *
 * Linearizability everywhere:
 *
 *     More coordination
 *          |
 *          v
 *     More latency
 *          |
 *          v
 *     More cost
 *
 *
 * Many operations ko actually latest value
 * immediately nahi chahiye.
 *
 *
 * Example:
 *
 *     Search result 1 second old
 *
 * Usually fine.
 *
 *
 * But:
 *
 *     Payment balance 1 second old
 *
 * Potentially dangerous.
 *
 *
 * ============================================================
 * 54. WHY NOT EVENTUAL CONSISTENCY EVERYWHERE?
 * ============================================================
 *
 * Eventual everywhere:
 *
 *     Less coordination
 *          |
 *          v
 *     But correctness bugs
 *          |
 *          v
 *     Application code mein shift
 *
 *
 * Example:
 *
 *     Inventory = 1
 *
 *     Two replicas:
 *
 *         Both think stock = 1
 *
 *     Both approve reservation.
 *
 *     Overselling.
 *
 *
 * Therefore:
 *
 *     Strong consistency only where invariants
 *     demand it.
 *
 *     Weak consistency where staleness is acceptable.
 *
 *
 * ============================================================
 * 55. MOST IMPORTANT DIFFERENCES
 * ============================================================
 *
 *
 * LINEARIZABILITY
 *
 *     Scope:
 *         Single object / operation
 *
 *     Real-time:
 *         YES
 *
 *     Main idea:
 *         Latest completed write must be visible
 *         to later reads.
 *
 *
 * ------------------------------------------------------------
 *
 * SEQUENTIAL CONSISTENCY
 *
 *     Scope:
 *         Operations
 *
 *     Real-time:
 *         NOT required
 *
 *     Main idea:
 *         One valid global order + each client's
 *         local order preserved.
 *
 *
 * ------------------------------------------------------------
 *
 * SERIALIZABILITY
 *
 *     Scope:
 *         Transactions
 *
 *     Real-time:
 *         NOT necessarily
 *
 *     Main idea:
 *         Concurrent transactions behave like
 *         some serial execution.
 *
 *
 * ------------------------------------------------------------
 *
 * STRICT SERIALIZABILITY
 *
 *     Scope:
 *         Transactions
 *
 *     Real-time:
 *         YES
 *
 *     Main idea:
 *         Serializability + real-time order.
 *
 *
 * ------------------------------------------------------------
 *
 * EVENTUAL
 *
 *     Scope:
 *         Replica convergence
 *
 *     Real-time:
 *         NO
 *
 *     Main idea:
 *         If updates stop and replication succeeds,
 *         replicas eventually converge.
 *
 *
 * ------------------------------------------------------------
 *
 * CAUSAL
 *
 *     Scope:
 *         Related operations
 *
 *     Real-time:
 *         Cause-effect order preserved
 *
 *     Main idea:
 *         Effect cannot appear before its cause.
 *
 *
 * ------------------------------------------------------------
 *
 * READ-YOUR-WRITES
 *
 *     Scope:
 *         One client/session
 *
 *     Main idea:
 *         User must see own writes.
 *
 *
 * ------------------------------------------------------------
 *
 * MONOTONIC READS
 *
 *     Scope:
 *         One client/session
 *
 *     Main idea:
 *         Data must not move backward for user.
 *
 *
 * ------------------------------------------------------------
 *
 * MONOTONIC WRITES
 *
 *     Scope:
 *         One client's writes
 *
 *     Main idea:
 *         Writes preserve client order.
 *
 *
 * ------------------------------------------------------------
 *
 * WRITES-FOLLOW-READS
 *
 *     Scope:
 *         Read -> dependent write
 *
 *     Main idea:
 *         Write must be ordered after state it read.
 *
 *
 * ============================================================
 * 56. INTERVIEW QUICK ANSWERS
 * ============================================================
 *
 * Q1. What is a consistency model?
 *
 * A:
 *     It is a contract defining which values a read
 *     is allowed to return after a history of writes.
 *
 *
 * ------------------------------------------------------------
 *
 * Q2. What is linearizability?
 *
 * A:
 *     A strong consistency guarantee where operations
 *     appear atomic and real-time ordering is preserved.
 *
 *
 * ------------------------------------------------------------
 *
 * Q3. Linearizability vs serializability?
 *
 * A:
 *     Linearizability is mainly about individual
 *     operations/objects and real-time order.
 *
 *     Serializability is about transactions and
 *     does not necessarily preserve real-time order.
 *
 *
 * ------------------------------------------------------------
 *
 * Q4. What is strict serializability?
 *
 * A:
 *     Serializability + real-time ordering.
 *
 *
 * ------------------------------------------------------------
 *
 * Q5. What is eventual consistency?
 *
 * A:
 *     If updates stop and replication succeeds,
 *     replicas eventually converge to the same value.
 *
 *
 * ------------------------------------------------------------
 *
 * Q6. Does eventual consistency guarantee latest reads?
 *
 * A:
 *     No.
 *
 *
 * ------------------------------------------------------------
 *
 * Q7. Does eventual consistency guarantee durability?
 *
 * A:
 *     No.
 *
 *     Durability is a separate property.
 *
 *
 * ------------------------------------------------------------
 *
 * Q8. What is causal consistency?
 *
 * A:
 *     It preserves cause-effect ordering.
 *     If B depends on A, anyone seeing B must
 *     see A first.
 *
 *
 * ------------------------------------------------------------
 *
 * Q9. What is read-your-writes?
 *
 * A:
 *     After a client writes a value, its later reads
 *     must see that value or a newer one.
 *
 *
 * ------------------------------------------------------------
 *
 * Q10. What are monotonic reads?
 *
 * A:
 *     Once a client observes a value, later reads
 *     cannot return an older value.
 *
 *
 * ------------------------------------------------------------
 *
 * Q11. Why use eventual consistency for search?
 *
 * A:
 *     Search is usually derived data and a small
 *     indexing delay is acceptable.
 *
 *
 * ------------------------------------------------------------
 *
 * Q12. Why strong consistency for inventory?
 *
 * A:
 *     Stale stock can cause overselling and violate
 *     a business invariant.
 *
 *
 * ------------------------------------------------------------
 *
 * Q13. Why causal consistency for chat?
 *
 * A:
 *     A reply should not appear before the message
 *     that caused the reply.
 *
 *
 * ------------------------------------------------------------
 *
 * Q14. Should an entire application use one model?
 *
 * A:
 *     No.
 *
 *     Production systems commonly mix models
 *     according to correctness requirements.
 *
 *
 * ============================================================
 * 57. GOLDEN SYSTEM-DESIGN RULE
 * ============================================================
 *
 * Don't ask:
 *
 *     "Which consistency model is best?"
 *
 *
 * Ask:
 *
 *     "What is the weakest guarantee that still
 *      prevents a correctness bug?"
 *
 *
 * Then:
 *
 *     Strong consistency
 *          |
 *          v
 *     Critical invariants
 *
 *
 *     Weaker consistency
 *          |
 *          v
 *     Derived / delay-tolerant data
 *
 *
 * ============================================================
 * 58. FINAL MEMORY MAP
 * ============================================================
 *
 *
 * STRONG
 *     |
 *     +--> Linearizability
 *     |       "Latest completed write visible."
 *     |
 *     +--> Strict Serializability
 *             "Transactions + real-time order."
 *
 *
 * INTERMEDIATE
 *     |
 *     +--> Causal
 *     |       "Cause before effect."
 *     |
 *     +--> Sequential
 *             "One valid global order."
 *
 *
 * WEAK
 *     |
 *     +--> Eventual
 *             "Eventually converge."
 *
 *
 * CLIENT-CENTRIC
 *     |
 *     +--> Read-Your-Writes
 *     |       "Mera change mujhe dikhe."
 *     |
 *     +--> Monotonic Reads
 *     |       "Data backward nahi jayega."
 *     |
 *     +--> Monotonic Writes
 *     |       "Mere writes same order mein."
 *     |
 *     +--> Writes-Follow-Reads
 *             "Read ke baad dependent write."
 *
 *
 * ============================================================
 * 59. ONE-LINE MEMORY TRICKS
 * ============================================================
 *
 * Linearizability:
 *
 *     "Write complete hua -> later read old value
 *      nahi de sakta."
 *
 *
 * Sequential:
 *
 *     "Ek valid order hona chahiye,
 *      real-time zaroori nahi."
 *
 *
 * Serializability:
 *
 *     "Transactions aise dikhein jaise
 *      one-by-one execute hui."
 *
 *
 * Strict Serializability:
 *
 *     "Serializability + real-time."
 *
 *
 * Eventual:
 *
 *     "Abhi different ho sakta hai,
 *      eventually same ho jayega."
 *
 *
 * Causal:
 *
 *     "Cause pehle, effect baad mein."
 *
 *
 * Read-Your-Writes:
 *
 *     "Apna change mujhe dikhao."
 *
 *
 * Monotonic Reads:
 *
 *     "Jo newer dekh liya,
 *      older mat dikhao."
 *
 *
 * Monotonic Writes:
 *
 *     "Mere writes reorder mat karo."
 *
 *
 * Writes-Follow-Reads:
 *
 *     "Jis state ko read karke write kiya,
 *      write uske baad ordered ho."
 *
 *
 * ============================================================
 * 60. FINAL SYSTEM DESIGN APPROACH
 * ============================================================
 *
 * Step 1:
 *
 *     Identify SOURCE OF TRUTH.
 *
 *
 * Step 2:
 *
 *     Identify HARD INVARIANTS.
 *
 *
 * Step 3:
 *
 *     Ask:
 *
 *         "What happens if read is stale?"
 *
 *
 * Step 4:
 *
 *     If stale data breaks correctness:
 *
 *         Strong consistency
 *
 *
 * Step 5:
 *
 *     If user mainly needs to see
 *     their own changes:
 *
 *         Read-Your-Writes
 *
 *
 * Step 6:
 *
 *     If cause-effect order matters:
 *
 *         Causal consistency
 *
 *
 * Step 7:
 *
 *     If data is derived and lag is acceptable:
 *
 *         Eventual consistency
 *
 *
 * Step 8:
 *
 *     Define:
 *
 *         - Maximum acceptable staleness
 *         - Conflict resolution
 *         - Failover behavior
 *         - Replication lag monitoring
 *         - Cache age
 *         - Consumer lag
 *
 *
 * ============================================================
 * FINAL RULE:
 *
 *     "Choose the weakest consistency model that still
 *      protects correctness."
 *
 *     Then measure the gap it leaves:
 *
 *         Replication lag
 *         Consumer lag
 *         Cache age
 *         Conflict rate
 *         Stale reads
 *         Failover behavior
 *
 * ============================================================
 */

In [ ]:
/**
 * CONSISTENCY MODELS
 *
 * Distributed system mein same data ki multiple copies hoti hain:
 *   Primary DB, Read Replicas, Cache, Search Index, Vector Index, Regional Replicas
 *
 * Replication ke benefits:
 *   - Higher availability
 *   - More read capacity
 *   - Lower geographic latency
 *   - Fault tolerance
 *
 * But ek important problem create hoti hai:
 *   "Write successful hone ke baad, later read ko kya value return karni chahiye?"
 *
 * Is rule ko CONSISTENCY MODEL kehte hain.
 *
 * IMPORTANT: Consistency ka matlab ye nahi ki data business-wise correct hai.
 * It only defines: "Storage system reads/writes ko kis order/visibility ke
 * saath expose karega?"
 */

/**
 * 1. WHY CONSISTENCY IS HARD
 *    Single machine: Client -> Database. One copy, one storage engine,
 *    easier ordering.
 *
 *    Distributed system: Client -> Node 1 --> Node 2 / Node 3
 *
 *    Ab different problems ho sakti hain:
 *      1. Write ek replica par pehle pahunch sakti hai.
 *      2. Read stale replica se aa sakti hai.
 *      3. Network partition replication rok sakta hai.
 *      4. Multiple regions same item ko simultaneously update kar sakti hain.
 *      5. Cache/search/vector index source DB se behind ho sakta hai.
 */

/**
 * 2. SIMPLE REPLICATION PROBLEM
 *    Suppose X = 3. Client: Write X = 5.
 *
 *      Client --Write X=5--> Node 1
 *        Node 1 --------------> Node 3   (X = 5)
 *        Node 1 -- delayed ---> Node 2   (X = 3, stale)
 *
 *    Ab another client Node 2 se read karta hai. Should it return:
 *      X = 3? | wait until Node 2 catches up? | read another replica? | reject?
 *
 *    Different consistency models ka answer different hota hai.
 */

/**
 * 3. CONSISTENCY IS A CONTRACT
 *    Consistency model define karta hai:
 *    "Given a history of writes, which values can a read return?"
 *
 *    "Database consistent hai" useful complete statement nahi hai.
 *    Instead ask: "Exactly kya guarantee mil rahi hai?"
 *
 *    Examples: Linearizability, Sequential Consistency, Strict Serializability,
 *    Causal Consistency, Read-Your-Writes, Monotonic Reads, Eventual Consistency
 */

/**
 * 4. CONSISTENCY SPECTRUM
 *
 *    WEAKER
 *      Eventual
 *      FIFO / PRAM
 *      Causal
 *      Client-centric guarantees
 *      Sequential
 *      Linearizable
 *      Strict Serializability
 *    STRONGER
 *
 *    Stronger guarantee -> more coordination -> more latency / complexity.
 *    Weaker guarantee   -> less coordination -> better availability / latency
 *                          -> more possibility of stale data.
 *
 *    IMPORTANT: some models strictly comparable nahi hain.
 *    Read-Your-Writes aur Monotonic Reads mainly client/session guarantees hain,
 *    jabki Linearizability system-wide guarantee hai.
 */

/**
 * 5. COORDINATION COST
 *    Strong consistency generally requires more coordination between nodes.
 *
 *      Client -> Node 1 --> Node 2 / Node 3 -> Confirm -> Client
 *
 *    Benefit: latest state ka confidence higher.
 *    Cost:
 *      - Higher latency
 *      - More network communication
 *      - Lower availability during partitions
 *      - Leader bottleneck
 *      - More conflicts
 *      - More complex failover
 *
 *    Weaker consistency: Client -> Local Replica -> Fast response.
 *    But replica stale ho sakti hai.
 */

/**
 * 6. RIGHT CONSISTENCY MODEL
 *    Golden rule: choose the WEAKEST consistency model that still protects
 *    correctness. Strongest consistency everywhere use mat karo.
 *
 *    Payment -> Strong | Inventory -> Strong
 *    Search -> Eventual | Recommendations -> Eventual | Analytics -> Eventual
 */

/**
 * 7. LINEARIZABILITY
 *    Strong consistency model.
 *    Simple meaning: system aise behave kare jaise ek hi latest
 *    authoritative copy ho.
 *
 *    Main rule: agar WRITE READ start hone se pehle complete ho gayi,
 *    to READ ko woh WRITE ya newer value dikhni chahiye.
 *
 *    Example: Initial X = 0.
 *      Client 1: Write X = 1 -> OK
 *      Client 2: Read X -> X = 1 (NOT X = 0)
 */

/**
 * 8. LINEARIZATION POINT
 *    Har operation ko imagine karo ki ek exact point par atomically execute hui.
 *
 *      Request -> [ Write ... X ... ] -> Response
 *                              ^
 *                      Linearization Point
 *
 *    Operation ke request-response interval ke andar ek point hota hai jahan
 *    operation logically take effect karti hai.
 */

/**
 * 9. LINEARIZABILITY GUARANTEES
 *    1. Real-time ordering
 *       A complete hua, phir B start hua -> B ko A ya uske baad ka state
 *       dekhna hoga.
 *    2. Atomic visibility
 *       Operation ek point par visible hoti hai.
 *    3. Single-object clarity
 *       Client replicated object ko ek single copy ki tarah reason kar sakta hai.
 */

/**
 * 10. LINEARIZABILITY IMPLEMENTATION
 *     Common techniques: leader, consensus, quorum, leader-freshness verification.
 *
 *       Client -> Leader --> Replica 1 / Replica 2
 *
 *     Follower ke disk par stale data ho sakta hai. Strong read guarantee
 *     maintain karne ke liye system unsafe follower read serve nahi karega.
 */

/**
 * 11. LINEARIZABILITY USE CASES
 *     Use when stale data correctness bug create kar sakta hai.
 *
 *     Examples: distributed locks, leader election metadata, account balance
 *     reads, inventory reservation, quota counters, permission changes,
 *     idempotency records, critical state transitions.
 *
 *     AI systems mein: billing usage, rate limits, model rollout state,
 *     access control, prompt safety policy versions, long-running job ownership.
 *
 *     Usually unnecessary for: recommendations, embedding lookup,
 *     analytics counters.
 */

/**
 * 12. SEQUENTIAL CONSISTENCY
 *     Linearizability se weaker.
 *
 *     Rule: all operations kisi ek sequential order mein appear honi chahiye.
 *     BUT global order ko real-time order follow karna mandatory nahi.
 *     Important: each client's own operation order preserve hona chahiye.
 *
 *     Example:
 *       Client 1: Write X = 1, Read Y = 0
 *       Client 2: Write Y = 1, Read X = 1
 *     Ye ek valid sequential ordering se explain ho sakta hai, as long as
 *     each client's local order preserved hai.
 */

/**
 * 13. LINEARIZABLE vs SEQUENTIAL
 *     LINEARIZABLE: real-time order MUST be respected.
 *       Write completes -> Read starts -> Read must see write.
 *     SEQUENTIAL: single global sequential order hona chahiye, but real-time
 *       order necessarily preserve nahi karna.
 *
 *     Memory trick: Linearizable = Sequential + Real-Time
 */

/**
 * 14. SERIALIZABILITY
 *     Transactions ke liye isolation guarantee.
 *
 *     Rule: concurrent transactions ka result kisi serial execution ke
 *     equivalent hona chahiye.
 *
 *     Transaction A aur B simultaneously execute kar sakti hain, but final
 *     result aisa dikhe: A -> B  OR  B -> A (one after another).
 */

/**
 * 15. SERIALIZABILITY != LINEARIZABILITY
 *     Serializability: transaction correctness.
 *       Real-time ordering necessarily required nahi.
 *
 *     Strict Serializability = Serializability + Real-time ordering.
 *       Transaction A commits -> Transaction B starts
 *       -> A must appear before B.
 */

/**
 * 16. IMPORTANT COMPARISON
 *     Linearizability        : single-object operations   | real-time order YES
 *     Serializability        : transactions               | real-time order NOT necessarily
 *     Strict Serializability : transactions               | real-time order YES
 */

/**
 * 17. STRICT SERIALIZABILITY USE CASES
 *     Strong transaction correctness required:
 *       financial transfers, seat booking, scarce inventory, order placement,
 *       entitlement grants, billing updates, multi-row invariants,
 *       multi-object correctness.
 */

/**
 * 18. EVENTUAL CONSISTENCY
 *     Weak consistency model.
 *
 *     Main promise: agar updates stop ho jayein AND replication succeed ho,
 *     to eventually all replicas same value par converge karengi.
 *
 *     Example: Write X = 5
 *       Now  : Replica 1 = 5, Replica 2 = 0, Replica 3 = 0
 *       Later: Replica 1 = 5, Replica 2 = 5, Replica 3 = 5
 */

/**
 * 19. WHAT EVENTUAL CONSISTENCY DOES NOT PROMISE
 *     It does NOT guarantee:
 *       - When convergence will happen
 *       - Latest write immediately visible
 *       - Bounded staleness
 *       - Read-your-writes
 *       - Monotonic reads
 *       - Conflict resolution
 *       - Durability
 *
 *     VERY IMPORTANT:
 *       Eventual consistency != Durable
 *       Eventual consistency != Automatically available
 *
 *     It mainly describes: Visibility + Convergence
 */

/**
 * 20. EVENTUAL CONSISTENCY EXAMPLE
 *
 *       Writer -> Replica 1 -- delayed --> Replica 2
 *                           -- delayed --> Replica 3
 *
 *     Immediately: R1 = 5, R2 = 0, R3 = 0
 *     Later      : R1 = 5, R2 = 5, R3 = 5
 *     Convergence achieved.
 */

/**
 * 21. CONFLICT RESOLUTION
 *     Eventual consistency mein multiple replicas independently write kar sakti hain.
 *
 *     Example: Replica 1: Name = "Mohammad", Replica 2: Name = "Mohd"
 *     Network partition ke baad: Conflict!
 *     System ko decide karna padega: which value wins?
 */

/**
 * 22. LAST-WRITE-WINS (LWW)
 *     Rule: highest timestamp/version wins.
 *
 *     Example:
 *       Write A: X = 10, timestamp = 100
 *       Write B: X = 20, timestamp = 101
 *       Result : X = 20
 *
 *     Problem: valid write silently lose ho sakti hai.
 *     Good fit   : ephemeral presence, cache refresh.
 *     Risky for  : profiles, documents, shopping carts, user data.
 */

/**
 * 23. VERSION VECTORS / VECTOR CLOCKS
 *     Purpose: causality track karna aur concurrent updates detect karna.
 *
 *     BUT conflict detect karne ke baad bhi application ko merge logic ki
 *     zarurat ho sakti hai.
 */

/**
 * 24. CRDT
 *     CRDT = Conflict-free Replicated Data Type.
 *
 *     Idea: aise data types design karo jo coordination ke bina safely
 *     merge ho sakein.
 *     Benefit    : automatic merge possible.
 *     Limitation : har type of operation/data ke liye suitable nahi.
 */

/**
 * 25. APPLICATION-LEVEL MERGE
 *     Business logic decide kare: Region A + Region B -> Custom merge.
 *
 *     Benefit: business correctness preserve kar sakte hain.
 *     Cost   : build aur test karna difficult.
 */

/**
 * 26. REJECT ON CONFLICT
 *     Conflict detect karo: version mismatch -> reject -> client retries.
 *
 *     Benefit: silent data loss avoid hota hai.
 *     Cost   : user ko failures/retries dikh sakte hain.
 */

/**
 * 27. WHERE EVENTUAL CONSISTENCY WORKS WELL
 *     Good use cases: CDN caches, DNS, like/view counters, search indexes,
 *     vector indexes, feature stores, analytics, activity feeds,
 *     notifications, recommendations.
 *
 *     Why? Slight delay usually acceptable hai.
 *
 *     Example: Document uploaded -> Embedding created -> Vector index updated.
 *     User ko immediately search result na mile, generally acceptable.
 */

/**
 * 28. WHERE EVENTUAL CONSISTENCY IS DANGEROUS
 *     Avoid for hard invariants: inventory reservation, payment authorization,
 *     access control, safety policy updates, exclusive job ownership.
 *
 *     Example: Stock = 1
 *       Region A -> Reserve SUCCESS, Region B -> Reserve SUCCESS
 *       Result: 1 item, 2 customers. BAD!
 */

/**
 * 29. SOURCE OF TRUTH vs DERIVED DATA
 *     SOURCE OF TRUTH -> Strong consistency.
 *     DERIVED DATA    -> Weaker consistency.
 *
 *       Primary DB --+--> Search Index
 *                    +--> Vector Index
 *                    +--> Analytics
 *                    +--> Recommendations
 *
 *     Primary DB    : stronger guarantees.
 *     Derived systems: eventual consistency often acceptable.
 */

/**
 * 30. CAUSAL CONSISTENCY
 *     Preserves CAUSE -> EFFECT.
 *     Agar B depends on A, to jisne B dekha usne A pehle dekha hoga.
 *
 *     Example:
 *       Alice: "Let's meet at 5 PM"
 *       Bob reads Alice's message, then replies: "Sounds good"
 *
 *     Bob's reply Alice's message par depend karta hai.
 *     Order: Alice's message -> Bob's reply.
 *     Must NOT appear as: Bob's reply -> Alice's message.
 */

/**
 * 31. SOURCES OF CAUSALITY
 *     1. Program order: same client's execution mein A happens before B.
 *     2. Read-from relationship: client reads A, phir A ke basis par B write karta hai.
 *     3. Transitivity: A -> B and B -> C, therefore A -> C.
 */

/**
 * 32. CAUSAL CONSISTENCY USE CASES
 *     Social feeds, comment threads, chat, collaborative documents,
 *     issue trackers, code review, notebook collaboration,
 *     AI assistant conversations.
 *
 *     Especially useful when user ko cause-effect order naturally expected hota hai.
 */

/**
 * 33. PRAM / FIFO CONSISTENCY
 *     Main guarantee: ek process ke writes ko every other process
 *     same order mein observe kare.
 *
 *     Example: Process 1: Write A, Write B -> others must observe A before B.
 *
 *     BUT different processes ke writes ko globally order karna necessary nahi.
 *     Also: read-based dependency preserve nahi hoti.
 */

/**
 * 34. PRAM vs CAUSAL
 *     Process 1: Write X = 1
 *     Process 2: Read X = 1, Write Y = 2
 *
 *     Causal: jisne Y = 2 dekha, usne X = 1 bhi dekha hoga.
 *     PRAM  : ye dependency guarantee nahi karta.
 *
 *     Therefore PRAM < Causal (is dependency guarantee ke terms mein).
 */

/**
 * 35. CLIENT-CENTRIC CONSISTENCY
 *     Kuch guarantees poore distributed system ke liye nahi, sirf ek
 *     client/session ke liye hoti hain.
 *
 *     Useful because user-facing bugs often session-related hote hain.
 *     Example: user profile photo change karta hai -> refresh -> old photo.
 *     Other users ko issue nahi, but same user ko confusing lagega.
 */

/**
 * 36. READ-YOUR-WRITES
 *     Guarantee: client ne jo write kiya, usi client ke later reads ko
 *     woh write ya newer value dikhni chahiye.
 *
 *     Example: Client writes X = 5 to Replica 1 (replication to Replica 2
 *     in progress). Client later reads X -> must return X = 5 or newer,
 *     NOT X = 3.
 */

/**
 * 37. READ-YOUR-WRITES IMPLEMENTATION
 *     Common techniques:
 *       - Route reads to primary temporarily
 *       - Session tokens
 *       - Sticky session / replica pinning
 *       - Minimum version on reads
 */

/**
 * 38. READ-YOUR-WRITES USE CASES
 *     Profile update, account settings, uploaded files, conversation messages,
 *     prompt edits, saved agents, workspace changes.
 *
 *     Simple rule: "User ne save kiya hai, to usko apna change immediately dikhna chahiye."
 */

/**
 * 39. MONOTONIC READS
 *     Guarantee: once client ne koi value observe kar li, later read older
 *     value return nahi karega.
 *
 *     Example: First read X = 5. Later another replica se read X = 3
 *     -> NOT ALLOWED. Later read must return X = 5, or newer, or wait.
 */

/**
 * 40. WITHOUT MONOTONIC READS
 *     User sees: Read 1 -> X = 5, refresh, Read 2 -> X = 3.
 *     Data BACKWARD in time move karta dikhta hai.
 *
 *     Can cause: user retries operation, support tickets, loss of trust,
 *     "data disappeared" feeling.
 *
 *     Good use cases: order history, message threads, audit logs,
 *     document versions, deployment history, evaluation runs.
 */

/**
 * 41. MONOTONIC WRITES
 *     Guarantee: one client's writes us client ke order mein apply hon.
 *
 *     Example: Client: Write A, Write B.
 *     Replica must NOT apply B then A. It must preserve A -> B.
 */

/**
 * 42. WHY MONOTONIC WRITES MATTER
 *     Later writes earlier writes par depend kar sakti hain.
 *
 *     Example 1: Model artifact upload -> Model configuration publish.
 *       Configuration artifact se pehle publish ho gaya: BAD.
 *     Example 2: Migration complete flag -> Read new schema.
 *       Flag pehle visible ho gaya, migration actually complete nahi hui: BAD.
 */

/**
 * 43. WRITES-FOLLOW-READS
 *     Guarantee: client ne value READ ki, phir us value ke basis par WRITE
 *     kiya -> WRITE us READ ke baad ordered honi chahiye.
 *
 *     Example: Read version = 7 -> client edits document -> Write version = 8.
 *     System ko ensure karna hai ki write actually version 7 ke state ke
 *     baad hai.
 */

/**
 * 44. WITHOUT WRITES-FOLLOW-READS
 *     Client: Read X = 7 -> Modify X -> Write.
 *     But system stale/incorrect ordering ki wajah se aisa write accept kar
 *     sakta hai jo client ke observed state par actually based nahi hai.
 *
 *     Useful for: compare-and-swap, conditional updates, collaborative editing,
 *     optimistic concurrency, workflow transitions, AI annotation/review systems.
 */

/**
 * 45. CLIENT-CENTRIC MODELS QUICK TABLE
 *     Read-Your-Writes    : "Mera write mujhe later read mein dikhe."
 *     Monotonic Reads     : "Jo newer value dekh li, uske baad older value mat dikhao."
 *     Monotonic Writes    : "Mere writes same order mein apply karo."
 *     Writes-Follow-Reads : "Mere read ke basis par hua write us read ke baad ordered ho."
 */

/**
 * 46. REAL DATABASES CONSISTENCY KO KAISE EXPOSE KARTE HAIN?
 *     Production database generally simple "Strong" / "Weak" buttons nahi deta.
 *
 *     Instead knobs milte hain:
 *       read concern, write concern, quorum, transaction isolation,
 *       leader reads, follower reads, session guarantees,
 *       global table modes, cache TTL.
 *
 *     IMPORTANT: "Is database consistent?" se better question:
 *       "Is particular operation ke liye, acknowledged write ko kaunse reads,
 *        kaunse region se, kaunse failure conditions mein observe karenge?"
 */

/**
 * 47. COMMON IMPLEMENTATION PATTERNS
 *
 *     SINGLE LEADER + PRIMARY READS
 *       Client -> Leader. Strong reads from leader. Followers may lag.
 *
 *     LEADER + ASYNC READ REPLICAS
 *       Client -> Read Replica. Fast/scalable, but may return stale data.
 *
 *     CONSENSUS GROUP
 *       Node 1 / Node 2 / Node 3 using Raft / Paxos.
 *       Can provide linearizable operations within the group.
 *
 *     QUORUM READ + WRITE
 *       N = total replicas, R = read quorum, W = write quorum.
 *       Common intuition: R + W > N means read/write quorums overlap,
 *       which can provide stronger visibility under appropriate failure
 *       assumptions.
 *       IMPORTANT: R + W > N alone magically har consistency property
 *       guarantee nahi karta. Failure handling and implementation matter.
 *
 *     SESSION TOKENS
 *       Useful for: read-your-writes, monotonic reads.
 *
 *     MULTI-REGION ACTIVE-ACTIVE
 *       Region A <----> Region B.
 *       Benefit: low local latency.
 *       Cost   : conflict resolution OR global coordination.
 *
 *     CACHE + TTL / INVALIDATION
 *       Primary DB -> Cache. Visibility may be eventual.
 *       Freshness depends on TTL and invalidation correctness.
 *
 *     EVENT-DRIVEN PROJECTION
 *       Source DB -> Event -> Consumer -> Derived View. Eventual consistency.
 *       Freshness depends on consumer lag and replay health.
 */

/**
 * 48. DON'T ASSUME CONSISTENCY FROM DATABASE TYPE
 *     Mat kaho: "NoSQL = Eventual" ya "SQL = Strong".
 *
 *     Actual guarantee depends on: product, deployment mode, operation,
 *     index type, read/write configuration, transaction mode.
 *
 *     Key-value store : may support eventual reads + selected strong reads.
 *     Wide-column store: may support quorum levels.
 *     Document DB     : may expose read/write concerns and sessions.
 *     Global SQL      : can provide strict transactions, but cross-region
 *                       writes may have coordination latency.
 *     Search / Vector Index: often derived from primary DB and intentionally lagging.
 */

/**
 * 49. HOW TO CHOOSE CONSISTENCY MODEL
 *     DON'T START WITH: "Which database should I use?"
 *     START WITH      : "What breaks if this read is stale?"
 *
 *     Ask:
 *       1.  What is the source of truth?
 *       2.  Which reads must observe acknowledged writes?
 *       3.  Is guarantee global, regional, per-session, per-object, etc.?
 *       4.  What happens during network partition?
 *       5.  Can two writers update same item?
 *       6.  If conflict occurs, who resolves it?
 *       7.  How much staleness is acceptable?
 *       8.  Does UI need pending/syncing status?
 *       9.  Are search/vector/cache/analytics clearly derived data?
 *       10. Do tests cover stale reads, reordered events, retries,
 *           duplicate delivery and failover?
 */

/**
 * 50. CONSISTENCY DECISION TREE
 *
 *     Can stale data be tolerated?
 *       NO  -> Strong model
 *       YES -> Must user see own changes?
 *                YES -> Read-Your-Writes
 *                NO  -> Must cause & effect stay ordered?
 *                         YES -> Causal
 *                         NO  -> Eventual
 */

/**
 * 51. USE CASE -> CONSISTENCY MODEL
 *
 *     PAYMENT AUTHORIZATION   : Strict Serializability
 *                               Money invariants preserve hone chahiye.
 *     BANK BALANCE AFTER TRANSFER: Linearizable / Transactionally consistent
 *                               Stale balance dangerous hai.
 *     SCARCE INVENTORY        : Linearizable claim OR Serializable transaction
 *                               Overselling prevent karna hai.
 *     DISTRIBUTED LOCK        : Linearizability
 *                               Do clients simultaneously lock hold nahi karne chahiye.
 *     PERMISSION REVOCATION   : Strong read on enforcement path
 *                               Stale permission security issue create kar sakti hai.
 *     USER PROFILE UPDATE     : Read-Your-Writes
 *                               User ko apna saved change dikhna chahiye.
 *     CHAT / COMMENTS         : Causal Consistency
 *                               Reply original message se pehle appear nahi hona chahiye.
 *     SEARCH                  : Eventual + measured lag
 *                               Search index derived data hai.
 *     VECTOR RETRIEVAL        : Eventual + ingestion status
 *                               Embedding/indexing asynchronous hai.
 *     LIKE / VIEW COUNTS      : Eventual / Approximate
 *                               Exact freshness usually worth the cost nahi.
 *     ANALYTICS DASHBOARD     : Eventual + freshness timestamp
 *                               User ko pata hona chahiye data kitna old hai.
 *     MODEL EVALUATION METRICS: Eventual + run state
 *                               Aggregates completed events se lag kar sakte hain.
 */

/**
 * 52. MIXED CONSISTENCY SYSTEM
 *     Real production system mein ek hi consistency model everywhere use nahi hota.
 *
 *     Example AI Platform:
 *       STRONG CONSISTENCY:
 *         Billing, Quotas, Access Control, Model Deployment State,
 *         Exclusive Job Claims
 *       CAUSAL / SESSION CONSISTENCY:
 *         Conversations, Prompt Editing, Human Review Workflow
 *       EVENTUAL CONSISTENCY:
 *         Search Index, Vector Index, Usage Aggregates, Recommendations
 */

/**
 * 53. WHY NOT STRONG CONSISTENCY EVERYWHERE?
 *     Linearizability everywhere: more coordination -> more latency -> more cost.
 *
 *     Many operations ko latest value immediately nahi chahiye.
 *       Search result 1 second old: usually fine.
 *       Payment balance 1 second old: potentially dangerous.
 */

/**
 * 54. WHY NOT EVENTUAL CONSISTENCY EVERYWHERE?
 *     Eventual everywhere: less coordination, but correctness bugs
 *     application code mein shift ho jate hain.
 *
 *     Example: Inventory = 1, two replicas dono sochti hain stock = 1,
 *     dono reservation approve kar deti hain -> overselling.
 *
 *     Therefore: strong consistency only where invariants demand it;
 *     weak consistency where staleness is acceptable.
 */

/**
 * 55. MOST IMPORTANT DIFFERENCES
 *
 *     LINEARIZABILITY
 *       Scope: single object / operation | Real-time: YES
 *       Latest completed write must be visible to later reads.
 *
 *     SEQUENTIAL CONSISTENCY
 *       Scope: operations | Real-time: NOT required
 *       One valid global order + each client's local order preserved.
 *
 *     SERIALIZABILITY
 *       Scope: transactions | Real-time: NOT necessarily
 *       Concurrent transactions behave like some serial execution.
 *
 *     STRICT SERIALIZABILITY
 *       Scope: transactions | Real-time: YES
 *       Serializability + real-time order.
 *
 *     EVENTUAL
 *       Scope: replica convergence | Real-time: NO
 *       If updates stop and replication succeeds, replicas eventually converge.
 *
 *     CAUSAL
 *       Scope: related operations | Cause-effect order preserved
 *       Effect cannot appear before its cause.
 *
 *     READ-YOUR-WRITES    : one client/session  - user must see own writes.
 *     MONOTONIC READS     : one client/session  - data must not move backward.
 *     MONOTONIC WRITES    : one client's writes - writes preserve client order.
 *     WRITES-FOLLOW-READS : read -> dependent write - write ordered after state it read.
 */

/**
 * 56. INTERVIEW QUICK ANSWERS
 *
 * Q1. What is a consistency model?
 * A : A contract defining which values a read is allowed to return after a
 *     history of writes.
 *
 * Q2. What is linearizability?
 * A : A strong consistency guarantee where operations appear atomic and
 *     real-time ordering is preserved.
 *
 * Q3. Linearizability vs serializability?
 * A : Linearizability is mainly about individual operations/objects and
 *     real-time order. Serializability is about transactions and does not
 *     necessarily preserve real-time order.
 *
 * Q4. What is strict serializability?
 * A : Serializability + real-time ordering.
 *
 * Q5. What is eventual consistency?
 * A : If updates stop and replication succeeds, replicas eventually converge
 *     to the same value.
 *
 * Q6. Does eventual consistency guarantee latest reads?
 * A : No.
 *
 * Q7. Does eventual consistency guarantee durability?
 * A : No. Durability is a separate property.
 *
 * Q8. What is causal consistency?
 * A : It preserves cause-effect ordering. If B depends on A, anyone seeing B
 *     must see A first.
 *
 * Q9. What is read-your-writes?
 * A : After a client writes a value, its later reads must see that value or
 *     a newer one.
 *
 * Q10. What are monotonic reads?
 * A : Once a client observes a value, later reads cannot return an older value.
 *
 * Q11. Why use eventual consistency for search?
 * A : Search is usually derived data and a small indexing delay is acceptable.
 *
 * Q12. Why strong consistency for inventory?
 * A : Stale stock can cause overselling and violate a business invariant.
 *
 * Q13. Why causal consistency for chat?
 * A : A reply should not appear before the message that caused the reply.
 *
 * Q14. Should an entire application use one model?
 * A : No. Production systems commonly mix models according to correctness
 *     requirements.
 */

/**
 * 57. GOLDEN SYSTEM-DESIGN RULE
 *     Don't ask: "Which consistency model is best?"
 *     Ask      : "What is the weakest guarantee that still prevents a
 *                 correctness bug?"
 *
 *     Strong consistency   -> critical invariants.
 *     Weaker consistency   -> derived / delay-tolerant data.
 */

/**
 * 58. FINAL MEMORY MAP
 *
 *     STRONG
 *       Linearizability        : "Latest completed write visible."
 *       Strict Serializability : "Transactions + real-time order."
 *     INTERMEDIATE
 *       Causal                 : "Cause before effect."
 *       Sequential             : "One valid global order."
 *     WEAK
 *       Eventual               : "Eventually converge."
 *     CLIENT-CENTRIC
 *       Read-Your-Writes       : "Mera change mujhe dikhe."
 *       Monotonic Reads        : "Data backward nahi jayega."
 *       Monotonic Writes       : "Mere writes same order mein."
 *       Writes-Follow-Reads    : "Read ke baad dependent write."
 */

/**
 * 59. ONE-LINE MEMORY TRICKS
 *
 *     Linearizability     : "Write complete hua -> later read old value nahi de sakta."
 *     Sequential          : "Ek valid order hona chahiye, real-time zaroori nahi."
 *     Serializability     : "Transactions aise dikhein jaise one-by-one execute hui."
 *     Strict Serializability: "Serializability + real-time."
 *     Eventual            : "Abhi different ho sakta hai, eventually same ho jayega."
 *     Causal              : "Cause pehle, effect baad mein."
 *     Read-Your-Writes    : "Apna change mujhe dikhao."
 *     Monotonic Reads     : "Jo newer dekh liya, older mat dikhao."
 *     Monotonic Writes    : "Mere writes reorder mat karo."
 *     Writes-Follow-Reads : "Jis state ko read karke write kiya, write uske baad ordered ho."
 */

/**
 * 60. FINAL SYSTEM DESIGN APPROACH
 *     Step 1: Identify SOURCE OF TRUTH.
 *     Step 2: Identify HARD INVARIANTS.
 *     Step 3: Ask "What happens if read is stale?"
 *     Step 4: Stale data correctness todta hai -> Strong consistency.
 *     Step 5: User ko mainly apne changes dikhne chahiye -> Read-Your-Writes.
 *     Step 6: Cause-effect order matter karta hai -> Causal consistency.
 *     Step 7: Data derived hai aur lag acceptable hai -> Eventual consistency.
 *     Step 8: Define: maximum acceptable staleness, conflict resolution,
 *             failover behavior, replication lag monitoring, cache age,
 *             consumer lag.
 */

/**
 * FINAL RULE
 *   "Choose the weakest consistency model that still protects correctness."
 *
 *   Then measure the gap it leaves: replication lag, consumer lag,
 *   cache age, conflict rate, stale reads, failover behavior.
 */